In [1]:
from Djuren import Djuren_surface
import numpy as np
import torch
import open3d as o3d

Using open3d version at ['/Users/romywilliamson/miniconda3/envs/bens/lib/python3.8/site-packages/open3d']


In [2]:
#surface_config = {'coarse_patches_id' : 'big-cube'}
#surface_config = {'coarse_patches_id' : 'igea500', 'degree' : 2}
surface_config = {'coarse_patches_id' : 'bob500', 'degree' : 2}
#surface_config = {'coarse_patches_id' : 'twisted_torus500', 'degree' : 2}
#surface_config = {'coarse_patches_id' : 'fertility500', 'degree' : 2}

degree = surface_config['degree']
name = surface_config['coarse_patches_id']

my_Djuren_surface = Djuren_surface(surface_config = surface_config, mesh_res=6)



my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d_coloured, mesh_o3d_plain, unblended_patches = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain])

o3d.io.write_triangle_mesh('data/surfaces/'+name+'-djuren.obj', mesh_o3d_plain)












Initialising Djuren surface on cpu
Assiging a halfedge to each vertex.
Preparing V_he for halfedge mesh
Prepared V_he for halfedge mesh.

Optimising V_he.
Finished optimising V_he.



/Users/romywilliamson/Documents/BNS/bns/Djuren.py:87: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1724557166878/work/torch/csrc/utils/tensor_new.cpp:281.)
  self.x = torch.tensor( self.base_facepatch.vertices)[:,:2]
/Users/romywilliamson/Documents/BNS/bns/Djuren.py:234: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  V_face = torch.tensor(self.V[verts, :], dtype=dtype, device=device)   # (3,3)
/Users/romywilliamson/Documents/BNS/bns/Djuren.py:282: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(

Using polynomial vtx functions


/Users/romywilliamson/Documents/BNS/bns/Djuren.py:468: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  output = torch.einsum('fsp,fpsc->fsc', torch.tensor(self.blend_weights, dtype=torch.float32), torch.tensor(self.vertex_funcs, dtype=torch.float32))


[Open3D WARNING] Write OBJ can not include triangle normals.


True

In [3]:
import trimesh
import numpy as np
import scipy.sparse as sparse
import igl
import matplotlib.pyplot as plt
from plyfile import PlyData, PlyElement

# --- The export logic from your working script ---
def write_custom_colour_ply_file(tm=None, colouringdict=None, filepath=None):
    print('Writing custom PLY to:', filepath)
    tm.export(filepath)
    p = PlyData.read(filepath)
    v = p.elements[0]
    f = p.elements[1]
    
    extra_data_specs = [(name, 'f8', (3,)) for name in colouringdict.keys()]
    a = np.empty(len(v.data), v.data.dtype.descr + extra_data_specs)
    
    for name in v.data.dtype.fields:
        a[name] = v[name]
    
    for colouringname, colouring in colouringdict.items():    
        a[colouringname] = colouring[:, :3]
    
    v = PlyElement.describe(a, 'vertex')
    p = PlyData([v, f], text=True)
    p.write(filepath)


def export_curvature_exact_custom(obj_file_path, output_ply_path, mapping_func, cmap):
    # 1. Load the OBJ
    print(f"Loading mesh: {obj_file_path}")
    tm = trimesh.load_mesh(obj_file_path)

    # 2. Compute EXACT Laplace-Beltrami operators using libigl
    print("Computing exact Cotangent Laplacian and Mass matrices...")
    L = igl.cotmatrix(tm.vertices, tm.faces)
    M = igl.massmatrix(tm.vertices, tm.faces, igl.MASSMATRIX_TYPE_BARYCENTRIC)

    # 3. Manually invert the Mass Matrix (Bypassing the igl.invert_diag error)
    m_diag = M.diagonal()
    m_inv = np.zeros_like(m_diag)
    
    # Safely invert only non-zero areas to prevent divide-by-zero errors
    valid_mask = m_diag > 1e-12
    m_inv[valid_mask] = 1.0 / m_diag[valid_mask]
    
    # Rebuild as a SciPy sparse diagonal matrix
    Minv = sparse.diags(m_inv)

    # 4. Calculate exact displacement (Delta x)
    # The true Laplace-Beltrami operator is Minv * L
    print("Applying operator to vertices...")
    delta_x = Minv.dot(L.dot(tm.vertices))

    # 5. Extract scalar Mean Curvature
    # IMPORTANT SIGN NOTE: libigl's cotmatrix convention makes L point INWARD. 
    # Depending on your colormap orientation, you might need to flip the sign here 
    # to match your PyTorch DiffMod (e.g., multiply by -0.5 instead of 0.5).
    mean_curvature = -0.5 * np.sum(delta_x * tm.vertex_normals, axis=1)

    # 6. Map to Colors (Float array for 'f8' export)
    print("Mapping colors...")
    flat_colors = cmap(mapping_func(mean_curvature))[:, :3].astype(np.float32) 

    # 7. Standard Visuals
    
    tm.visual.vertex_colors = (flat_colors * 255).astype(np.uint8)

    # 8. Export
    name_key = 'meancurv_colours'
    write_custom_colour_ply_file(
        tm=tm,
        colouringdict={name_key: flat_colors},
        filepath=str(output_ply_path)
    )
    
    tm.export(obj_file_path)
    print("Done!")


# --- Execution ---
if __name__ == "__main__":
    from colourmappings import mapping12
    
    # Configuration
    name = name # Replace with actual variable
    input_obj = f'data/surfaces/{name}-djuren.obj'
    output_ply = f'data/surfaces/{name}-djuren.ply'
    
    meancurv_cmap = plt.get_cmap('seismic')
    
    export_curvature_exact_custom(
        input_obj, 
        output_ply, 
        mapping_func=mapping12, 
        cmap=meancurv_cmap
    )

Loading mesh: data/surfaces/bob500-djuren.obj
Computing exact Cotangent Laplacian and Mass matrices...
Applying operator to vertices...
Mapping colors...
Writing custom PLY to: data/surfaces/bob500-djuren.ply
Done!


In [4]:







'''
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
#mesh_o3d, mesh_o3d_plain = my_Djuren_surface.build_output_surface(colour_flag='blend')
#o3d.visualization.draw_geometries([mesh_o3d])
mesh_o3d, mesh_o3d_plain = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d])
'''



'''
my_Djuren_surface.compute_blend_weights(flag='bary') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries(unblended_o3d)
#o3d.visualization.draw_geometries([mesh_o3d])
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, djuren polynomials")
'''

'''

my_Djuren_surface.compute_blend_weights(flag='bary') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal') #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries(unblended_o3d)
#o3d.visualization.draw_geometries([mesh_o3d])
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, equal-angle polynomials")
'''




'''
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="djuren blend, djuren polynomials")
#o3d.visualization.draw_geometries([mesh_o3d],  window_name="djuren blend, djuren polynomials")
'''

'''
my_Djuren_surface.compute_blend_weights(flag='inv-exp', overlap_param = 0.99) #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="inv exp blend, equal-angle polynomials")
o3d.visualization.draw_geometries([mesh_o3d],  window_name="inv exp blend, inv exp polynomials")



my_Djuren_surface.compute_blend_weights(flag='inv-exp') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="inv-exp blend, djuren polynomials")
#o3d.visualization.draw_geometries([mesh_o3d],  window_name="djuren blend, djuren polynomials")

my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='blend')
o3d.visualization.draw_geometries([mesh_o3d_plain],  window_name="djuren blend, equal-angle polynomials")
#o3d.visualization.draw_geometries([mesh_o3d],  window_name="inv exp blend, equal-angle polynomials")

my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="djuren blend, equal-angle polynomials")

my_Djuren_surface.compute_blend_weights(flag='bary') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, djuren polynomials")

my_Djuren_surface.compute_blend_weights(flag='bary') #blend type
my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal', degree=degree) #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries([mesh_o3d_plain], window_name="bary blend, equal-angle polynomials")
'''

#my_Djuren_surface.he_mesh.paint_uniform_color([0, 0, 1])
#for i in range(3*my_Djuren_surface.F.shape[0]):
#    o3d.visualization.draw_geometries([unblended_o3d[i]] + [my_Djuren_surface.he_mesh])

#o3d.visualization.draw_geometries([mesh_o3d])
#o3d.visualization.draw_geometries(unblended_o3d + [my_Djuren_surface.he_mesh])
#o3d.visualization.draw_geometries([mesh_o3d])
#my_Djuren_surface.he_mesh.paint_uniform_color([0, 0, 1])


'''
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d, mesh_o3d_plain, unblended_o3d = my_Djuren_surface.build_output_surface(colour_flag='angles')
o3d.visualization.draw_geometries(unblended_o3d)
o3d.visualization.draw_geometries([mesh_o3d])
o3d.visualization.draw_geometries([mesh_o3d_plain])
'''


'''
##### Ours: smooth.
my_Djuren_surface.compute_blend_weights(flag='inv-exp') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal') #vertex func type, onering coords type
mesh_o3d_angles, mesh_o3d_plain = my_Djuren_surface.build_output_surface()

o3d.visualization.draw_geometries([mesh_o3d_angles])
o3d.visualization.draw_geometries([mesh_o3d_plain])

##### Ours with their blending: not smooth.
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='polynomial', onering_coords_flag='equal') #vertex func type, onering coords type
mesh_o3d_angles, mesh_o3d_plain = my_Djuren_surface.build_output_surface()

o3d.visualization.draw_geometries([mesh_o3d_angles])
o3d.visualization.draw_geometries([mesh_o3d_plain])
'''

'''
##### Theirs with their blending
my_Djuren_surface.compute_blend_weights(flag='djuren') #blend type

my_Djuren_surface.forward(function_flag='constant', onering_coords_flag='djuren') #vertex func type, onering coords type
mesh_o3d_angles, mesh_o3d_plain = my_Djuren_surface.build_output_surface()

o3d.visualization.draw_geometries([mesh_o3d_angles])
o3d.visualization.draw_geometries([mesh_o3d_plain])
'''


"\n##### Theirs with their blending\nmy_Djuren_surface.compute_blend_weights(flag='djuren') #blend type\n\nmy_Djuren_surface.forward(function_flag='constant', onering_coords_flag='djuren') #vertex func type, onering coords type\nmesh_o3d_angles, mesh_o3d_plain = my_Djuren_surface.build_output_surface()\n\no3d.visualization.draw_geometries([mesh_o3d_angles])\no3d.visualization.draw_geometries([mesh_o3d_plain])\n"

In [5]:

print([ my_Djuren_surface.bary_weight(torch.tensor([0.0,0.0]).unsqueeze(0), 0),
my_Djuren_surface.bary_weight(torch.tensor([0.0,0.0]).unsqueeze(0), 1),
my_Djuren_surface.bary_weight(torch.tensor([0.0,0.0]).unsqueeze(0), 2) ])

print([ my_Djuren_surface.bary_weight(torch.tensor([1.0,0.0]).unsqueeze(0), 0),
my_Djuren_surface.bary_weight(torch.tensor([1.0,0.0]).unsqueeze(0), 1),
my_Djuren_surface.bary_weight(torch.tensor([1.0,0.0]).unsqueeze(0), 2) ])


print([ my_Djuren_surface.bary_weight(torch.tensor([0.5,np.sqrt(3)/2]).unsqueeze(0), 0),
my_Djuren_surface.bary_weight(torch.tensor([0.5,np.sqrt(3)/2]).unsqueeze(0), 1),
my_Djuren_surface.bary_weight(torch.tensor([0.5,np.sqrt(3)/2]).unsqueeze(0), 2) ])

[tensor([1.]), tensor([0.]), tensor([0.])]
[tensor([0.]), tensor([1.]), tensor([0.])]
[tensor([8.9741e-09], dtype=torch.float64), tensor([8.9741e-09], dtype=torch.float64), tensor([1.], dtype=torch.float64)]
